# L3: Agent Loop

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

from deepagent_helpers import (WORKSPACE_DIR, ANALYSIS_PATH, 
    DEFAULT_TICKET_ID, langsmith_status, build_review_agent, get_ticket,
    run_review_loop_live, save_deepagent_trace, load_deepagent_trace)

from visualization_helpers import (render_review_trace, render_report, 
    render_requirement_comparison, plot_agent_tools)

%matplotlib inline
print("✓ Environment ready")

✓ Environment ready


<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> file:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.</p>

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>
</div>

## Requirements Review Agent

In [2]:
print(Path("workspace/memories/AGENTS.md").read_text())

# Requirements Review Agent

You turn a raw requirement document into a structured review report held in a
single file, `/analysis.md`.

## Procedure

Work through these stages in order, one tool call per turn. Do not skip a stage
and do not combine two stages into one call.

1. Call `fetch_requirement` to get the requirement document.
2. Call `write_file` to create `/analysis.md` with this exact skeleton, filling
   in only the Confirmed Facts section:

```
# Requirements Analysis

## Confirmed Facts
- one bullet per fact

## Ambiguities

## Clarification Questions

## Assumptions

## Summary
```

3. Call `edit_file` to fill in the Ambiguities section. One bullet per clause, in
   this shape:

```
- **"the exact clause"**
  Reason: why it is under-specified
```

4. Call `edit_file` to fill in the Clarification Questions section. One bullet each.
5. Call `edit_file` to fill in the Assumptions section. One bullet each.
6. Call `edit_file` to fill in the Summary section with two or three

In [3]:
MODEL = "openai:gpt-5.4-mini"
MAX_TURNS = 12

agent = build_review_agent(
    model = MODEL,
    max_turns = MAX_TURNS,
)

print("Agent created")

Agent created


| Tool | What it does |
|---|---|
| `fetch_requirement` | **Custom to this agent.** Fetches the requirement document. Simulates two transient failures before succeeding, so the Model has to retry. |
| `write_file` | Creates a new file, or overwrites one that already exists. Called once, to lay down the `analysis.md` skeleton. |
| `edit_file` | Makes an exact string replacement inside a file. Called once per remaining section, so each turn only touches the section it is filling in. |


## Execute the Agent Loop

In [4]:
TICKET_ID = DEFAULT_TICKET_ID
requirement_doc = get_ticket(TICKET_ID)

print(f"Ticket {TICKET_ID}:\n")
print(requirement_doc)

Ticket AUTH-142:


REQUIREMENT: User Profile File Upload

Users shall be able to upload files to their profile.

Uploaded files should be stored securely.

The system should support common file types.



<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> LLMs do not always produce the same results. Consequently, the agent behavior you experience when running labs may differ from the video.</p>

In [5]:
USE_LIVE_LLM = True
if USE_LIVE_LLM:
    run = run_review_loop_live(agent, ticket_id=TICKET_ID)
    save_deepagent_trace(run, "lesson_3_live_trace.json")
    print(f"✓ Ran live against {MODEL}, saved to lesson_3_live_trace.json")
    print(langsmith_status())
else:
    run = load_deepagent_trace("lesson_3_prebaked_trace.json")
    print("✓ Loaded a pre-baked real trace")

print(f"✓ Completed in {run.turns} model turns ({len(run.trace)} steps)")
print(f" {ANALYSIS_PATH}: {run.counts}, summary {run.summary_words} words")

✓ Ran live against openai:gpt-5.4-mini, saved to lesson_3_live_trace.json
✓ LangSmith tracing is ON — after a live run, go to https://smith.langchain.com, open Tracing in the left-hand menu, and select the project 'AgentLoop'.
✓ Completed in 9 model turns (17 steps)
 /analysis.md: {'Confirmed Facts': 3, 'Ambiguities': 3, 'Clarification Questions': 4, 'Assumptions': 3}, summary 46 words


### Execution Trace

Click on the following link to view saved agent execution trace.

🔗   [[ Agent's Trace ]](https://smith.langchain.com/public/1f4c8774-8714-42b4-a37f-8740849d78ed/r/01a10fbe-c6c0-78d3-83e2-14060a8e2ac7?scroll_to=input&start_time=2026-10-06T05%3A45%3A23.39242Z)

In [6]:
display(render_review_trace(run.trace))

Actor,Action,Result
🗨 Model reasoning,Calls fetch_requirement,—
⚙ Tool,fetch_requirement,REQUIREMENT: User Profile File Upload Users shall be able to upload files to their profile. Uploaded files should be stored securely. The system should support common file types.
🗨 Model reasoning,Calls write_file: Confirmed Facts,—
⚙ Tool,write_file: Confirmed Facts,Updated file /analysis.md
🗨 Model reasoning,Calls edit_file: Ambiguities,—
⚙ Tool,edit_file: Ambiguities,Successfully replaced 1 instance(s) of the string in '/analysis.md'
🗨 Model reasoning,Calls edit_file: Clarification Questions,—
⚙ Tool,edit_file: Clarification Questions,Successfully replaced 1 instance(s) of the string in '/analysis.md'
🗨 Model reasoning,Calls edit_file: Assumptions,—
⚙ Tool,edit_file: Assumptions,"Error: String '## Assumptions ' appears 2 times in file. Use replace_all=True to replace all instances, or provide a more specific string with surrounding context."


## The Report

In [7]:
display(render_requirement_comparison(
    requirement_doc, run.analysis_markdown, report_title="analysis.md"))